In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import RelaxedOneHotCategorical
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, entropy, ttest_ind
from sklearn.metrics import mean_absolute_error, r2_score, confusion_matrix

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (15, 10)

In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

def district_share_variance(district_votes: torch.Tensor, S: int) -> torch.Tensor:
    eps     = 1e-8
    #max_var = 1.0 / S
    max_var = 1.0
    totals  = district_votes.sum(dim=2, keepdim=True).clamp(min=eps)
    if (totals == 0).any():
        print("flag1!")
    share   = district_votes / totals
    mean    = share.mean(dim=2, keepdim=True)
    var     = ((share - mean) ** 2).mean(dim=2)
    return (var / max_var).clamp(0.0, 1.0)

def party_share_variance(district_votes: torch.Tensor, K: int) -> torch.Tensor:
    eps = 1e-8
    #max_var = 1.0 / K
    max_var = 1.0
    totals = district_votes.sum(dim=1, keepdim=True).clamp(min=eps)
    if (totals == 0).any():
        print("flag2!")
    shares = district_votes / totals
    mean = shares.mean(dim=1, keepdim=True)
    var = ((shares - mean) ** 2).mean(dim=1)
    return (var / max_var).clamp(0.0, 1.0)

def party_seat_share(district_votes: torch.Tensor) -> torch.Tensor:
    num_examples = district_votes.shape[0]
    num_districts = district_votes.shape[1]
    num_parties = district_votes.shape[2]
    theta = [];
    for i in range(num_examples):
        seats = np.zeros(num_parties)
        for j in range(num_districts):
            # Move the tensor to CPU and convert to NumPy array before using np.argmax
            votes = district_votes[i, j, :].detach().cpu().numpy()
            k = np.argmax(votes)
            seats[k] += 1
        theta.append(seats/num_districts)
    return np.array(theta)

In [34]:
class EncoderNN2(nn.Module):
    def __init__(self, num_districts, num_parties, hidden_dims=(256, 128, 64)):
    #def __init__(self, num_districts, num_parties, hidden_dims=(128, 64)):
        super().__init__()
        self.input_dim = num_districts * num_parties

        # Deeper network with batch normalization
        layers = []
        prev = self.input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.1))
            prev = h

        self.mlp = nn.Sequential(*layers)

        # Separate heads for alpha and beta
        self.alpha_head = nn.Sequential(
            nn.Linear(prev, 32),
            nn.ReLU(),
            nn.Linear(32, num_parties)
        )

        self.beta1_head = nn.Sequential(
            nn.Linear(prev, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

        self.beta2_head = nn.Sequential(
            nn.Linear(prev, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

        self.beta3_head = nn.Sequential(
            nn.Linear(prev, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, phi):

        x = phi.reshape(phi.shape[0], -1) / 1
        h = self.mlp(x)
        alpha  = F.softmax(10*self.alpha_head(h),  dim=1)   # (B, K)
        beta1 = 0.5+0.5*torch.sigmoid(self.beta1_head(h))
        beta2 = 0.5+0.5*torch.sigmoid(self.beta2_head(h))
        beta3 = 0.5+0.5*torch.sigmoid(self.beta3_head(h))
        #beta = np.stack((beta1, beta2, beta3), axis=1)
        #beta = torch.stack((beta1.squeeze(), beta2.squeeze(), beta3.squeeze()), dim=1)

        return alpha, beta1, beta2, beta3

In [38]:
import scipy

def _has_nan_grad(model):
    for p in model.parameters():
        if p.grad is not None and torch.isnan(p.grad).any():
            return True
    return False


#def train_model(data, K=3, S=50, N=5000, n_voters=500,
#                epochs=30, batch_size=32, lr=2e-4,
#                tau=1.0,
#                lambda_kl=0.5, lambda_ce=0.25):

def train_encoder_PCM(X, true_params, num_examples, num_districts, voters_per_district, num_parties,
                          device='cpu', epochs=50, batch_size=8, lr=1e-3, tau=0.5,
                          gumbel_temp=0.5, verbose=True):

    N = num_districts * voters_per_district
    K = num_parties
    S = num_districts

    encoder_model = EncoderNN2(num_districts=S, num_parties=K).to(device)
    optimizer = optim.AdamW(encoder_model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    mse_loss = nn.MSELoss()
    ce_loss = nn.CrossEntropyLoss()
    loss_history = {'total': [], 'reconstruction': [], 'variance': [], 'alpha': [], 'beta': [], 'theta': [], 'param_direct': []}
    best_loss = float('inf')

    alpha_true = torch.tensor(np.array([p[0] for p in true_params]), dtype=torch.float32, device=device)
    beta1_true = torch.tensor(np.array([p[1] for p in true_params]), dtype=torch.float32, device=device)
    beta2_true = torch.tensor(np.array([p[2] for p in true_params]), dtype=torch.float32, device=device)
    beta3_true = torch.tensor(np.array([p[3] for p in true_params]), dtype=torch.float32, device=device)

    start_time = time.time()
    indices = np.arange(num_examples)

    for ep in range(1, epochs + 1):
        encoder_model.train()
        np.random.shuffle(indices)
        epoch_loss_total = 0.0
        epoch_loss_recon = 0.0
        epoch_loss_alpha = 0.0
        epoch_loss_beta = 0.0
        epoch_loss_theta = 0.0
        epoch_loss_param = 0.0
        epoch_loss_variance = 0.0

        for i in range(0, num_examples, batch_size):
            batch_idx = indices[i:i+batch_size]
            X_range = []
            for j in batch_idx:
                x = X[j]
                distlist = np.random.choice(x.shape[0], num_districts, replace=False)
                x_sel = x[distlist]
                X_range.append(x_sel)

            phi_original = torch.tensor(np.stack(X_range), dtype=torch.float32, device=device)
            alpha_true_b = alpha_true[batch_idx]
            beta1_true_b = beta1_true[batch_idx]
            beta2_true_b = beta2_true[batch_idx]
            beta3_true_b = beta3_true[batch_idx]

            x_svar_true = district_share_variance(phi_original, num_districts)
            x_kvar_true = party_share_variance(phi_original, num_parties)
            theta_true = party_seat_share(phi_original)

            # Encoder predicts alpha, beta
            alpha_pred, beta1_pred, beta2_pred, beta3_pred = encoder_model(phi_original)

            #loss_recon += loss_variance

            # 2. Direct parameter supervision (helps with gradient flow)
            target_cls = torch.argmax(alpha_true_b, dim=1)
            #loss_alpha1_direct = ce_loss(alpha_logits, target_cls)  #predict winning party
            #loss_alpha2_direct = ce_loss(alpha_true_b, alpha_pred)
            loss_alpha3_direct = mse_loss(alpha_true_b, alpha_pred)
            loss_beta1_direct = mse_loss(beta1_true_b, beta1_pred)    #predict ABM parameter
            loss_beta2_direct = mse_loss(beta2_true_b, beta2_pred)
            loss_beta3_direct = mse_loss(beta3_true_b, beta3_pred)

            #loss_param_direct = loss_alpha1_direct + loss_alpha2_direct + loss_alpha3_direct + loss_beta_direct + loss_theta_direct
            loss = loss_beta1_direct + loss_beta2_direct + loss_beta3_direct + loss_alpha3_direct
            loss_beta_direct = loss_beta1_direct + loss_beta2_direct + loss_beta3_direct

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(encoder_model.parameters(), max_norm=1.0)
            optimizer.step()

            # Track losses
            epoch_loss_total += loss.item() * phi_original.shape[0]
            epoch_loss_alpha += loss_alpha3_direct.item() * phi_original.shape[0]
            epoch_loss_beta += loss_beta_direct.item() * phi_original.shape[0]

          # Average losses
        epoch_loss_total /= num_examples
        epoch_loss_recon /= num_examples
        epoch_loss_variance /= num_examples
        epoch_loss_param /= num_examples
        epoch_loss_alpha /= num_examples
        epoch_loss_beta /= num_examples
        epoch_loss_theta /= num_examples

        loss_history['total'].append(epoch_loss_total)
        loss_history['reconstruction'].append(epoch_loss_recon)
        loss_history['variance'].append(epoch_loss_variance)
        loss_history['param_direct'].append(epoch_loss_param)
        loss_history['alpha'].append(epoch_loss_alpha)
        loss_history['beta'].append(epoch_loss_beta)
        loss_history['theta'].append(epoch_loss_theta)

        # Update learning rate
        scheduler.step(epoch_loss_total)

        #if verbose and (ep % 10 == 0 or ep == 1):
        if verbose :
                print(f"Epoch {ep:04d} | Total: {epoch_loss_total:.4f} | "
                  f"Recon: {epoch_loss_recon:.4f} | Param: {epoch_loss_param:.4f} | "
                  f"Variance: {epoch_loss_variance:.4f} | "
                  f"Alpha: {epoch_loss_alpha:.4f} | Beta: {epoch_loss_beta:.4f} | "
                  f"Theta: {epoch_loss_theta:.4f} | "
                  f"LR: {optimizer.param_groups[0]['lr']:.6f} | "
                  f"elapsed {time.time()-start_time:.1f}s")

        # Save best model
        if epoch_loss_total < best_loss:
            best_loss = epoch_loss_total

        # Create X_tensor from the training data X (which has num_examples) for correlation calculation
        X_tensor_for_corr = torch.tensor(np.stack(X), dtype=torch.float32, device=device)
        alpha_pred_all, beta1_pred_all, beta2_pred_all, beta3_pred_all = encoder_model(X_tensor_for_corr)
        #c1 = np.correlate(alpha_pred_all.cpu().numpy(), alpha_true.cpu().numpy())
        c1 = scipy.stats.pearsonr(beta1_pred_all.detach().cpu().numpy().squeeze(), beta1_true.cpu().numpy().squeeze())[0]
        c2 = scipy.stats.pearsonr(beta2_pred_all.detach().cpu().numpy().squeeze(), beta2_true.cpu().numpy().squeeze())[0]
        c3 = scipy.stats.pearsonr(beta3_pred_all.detach().cpu().numpy().squeeze(), beta3_true.cpu().numpy().squeeze())[0]
        print("c1=", c1, "c2=", c2, "c3=", c3)

    return encoder_model, loss_history

In [35]:
import scipy
import scipy.io as sio
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)
mat_file_path = '/content/drive/MyDrive/indiasurveys_aug_PCM.mat'
data=sio.loadmat(mat_file_path)

X = data['CC']
alpha = data['alpha']
beta1 = data['beta1']
beta2 = data['beta2']
beta3 = data['beta3']
print(X.shape)
print(alpha.shape)
print(beta1.shape)
x= X[0][10]
print(x.shape)
print(alpha[500])
print(beta1[0])

NUM_DISTRICTS = X[0][0].shape[0]
NUM_PARTIES = X[0][0].shape[1]
VOTERS_PER_DISTRICT = 100 #int(np.sum(X[0][0][0]))

#voters_scaling = 100
#X = X/voters_scaling
#VOTERS_PER_DISTRICT = int(VOTERS_PER_DISTRICT/voters_scaling)

train_range = range(0,2500)
NUM_EXAMPLES = len(train_range)

X_np, Y, true_params = [], [], []
for i in train_range:
      X_np.append(X[0][i].astype(np.float32))
      # Fix: Removed float() conversion as beta[i] is an array, not a scalar
      Y.append((alpha[i].astype(np.float32), beta1[i].astype(np.float32), beta2[i].astype(np.float32), beta3[i].astype(np.float32)))
      true_params.append((alpha[i].astype(np.float32), beta1[i].astype(np.float32), beta2[i].astype(np.float32), beta3[i].astype(np.float32)))

Mounted at /content/drive
(1, 8000)
(8000, 3)
(5000, 1)
(10, 3)
[0.23825763 0.47528606 0.28645632]
[0.57225825]


In [39]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import RelaxedOneHotCategorical
import scipy.stats
import matplotlib.pyplot
import gc # Import garbage collector

from scipy.io import savemat
import numpy as np

EPOCHS = 50
BATCH_SIZE = 100  # Reduced batch size to mitigate OutOfMemoryError

#if __name__ == "__main__":
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

# Clear GPU memory before starting training, if available
if device == 'cuda':
    print("Clearing GPU memory...")
    torch.cuda.empty_cache()
    gc.collect() # Force Python garbage collection
    print("GPU memory cleared.")
    print("GPU memory summary AFTER clearing cache:")
    print(torch.cuda.memory_summary())

print(f"VOTERS_PER_DISTRICT: {VOTERS_PER_DISTRICT}")
print("Training on GPU..." if device == 'cuda' else "Training on CPU...")
encoder_PCM, loss_history = train_encoder_PCM(
      X_np, true_params,
      num_examples=NUM_EXAMPLES,
      num_districts=NUM_DISTRICTS,
      num_parties=NUM_PARTIES,
      voters_per_district=VOTERS_PER_DISTRICT,
      device=device,
      epochs=EPOCHS,
      batch_size=BATCH_SIZE,
      lr=1e-3,
      gumbel_temp=0.75
    )

print("\nEncoder training complete.")

Device: cuda
Clearing GPU memory...
GPU memory cleared.
GPU memory summary AFTER clearing cache:
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |  29510 KiB |   3837 MiB |   3134 GiB |   3134 GiB |
|       from large pool |  16640 KiB |     60 MiB |      3 GiB |      3 GiB |
|       from small pool |  12870 KiB |   3810 MiB |   3131 GiB |   3131 GiB |
|---------------------------------------------------------------------------|
| Active memory         |  29510 KiB |   3837

In [40]:
Xtest = data['CC']

X_test_processed = []
for i in range(Xtest[0].shape[0]): # Iterate through the actual number of samples in Xtest_raw[0]
    x = Xtest[0][i].astype(np.float32)
    X_test_processed.append(x)

X_test_processed = np.array(X_test_processed)
print(X_test_processed.shape)

X_tensor = torch.tensor(np.stack(X_test_processed), dtype=torch.float32, device=device)
alpha_pred, beta1_pred, beta2_pred, beta3_pred = encoder_PCM(X_tensor)
alpha_pred = alpha_pred.detach().cpu().numpy()
beta1_pred = beta1_pred.detach().cpu().numpy()
beta2_pred = beta2_pred.detach().cpu().numpy()
beta3_pred = beta3_pred.detach().cpu().numpy()

print(beta1[4000])
print(beta1_pred[4000])
print(alpha[4000])
print(alpha_pred[4000])

print(alpha_pred[5499])
print(alpha_pred[5999])
print(alpha_pred[6499])
print(alpha_pred[6999])
print(alpha_pred[7499])
print(alpha_pred[7999])

(8000, 10, 3)
[0.60252798]
[0.6646927]
[0.35591875 0.38981175 0.2542695 ]
[0.3043396  0.4053185  0.29034194]
[0.44726518 0.3956573  0.15707752]
[0.550408   0.2313179  0.21827407]
[0.35298938 0.51555336 0.13145728]
[0.32229763 0.24313481 0.4345675 ]
[0.44683158 0.39281747 0.16035098]
[0.430529   0.37353998 0.19593097]


In [33]:
def _gumbel_softmax_st(logits: torch.Tensor, tau: float) -> torch.Tensor:
    """Straight-through Gumbel-softmax.  logits: (..., C)  →  (..., C)."""
    eps    = 1e-9
    U      = torch.rand_like(logits).clamp(eps, 1 - eps)
    gumbel = -torch.log(-torch.log(U))
    y_soft = F.softmax((logits + gumbel) / tau, dim=-1)
    y_hard = torch.zeros_like(y_soft).scatter_(
        -1, y_soft.argmax(dim=-1, keepdim=True), 1.0
    )
    return (y_hard - y_soft).detach() + y_soft


class GradABMSimulator(nn.Module):
    def __init__(self, N: int, S: int, K: int,
                 n_voters: int = None, tau: float = 1.0, eps: float = 1e-6):
        super().__init__()
        self.N        = N
        self.S        = S
        self.K        = K
        self.n_voters = n_voters if n_voters is not None else N
        self.tau      = tau
        self.eps      = eps

    def forward(self, alpha: torch.Tensor, beta: torch.Tensor):

      B   = alpha.size(0)
      S, K, eps = self.S, self.K, self.eps
      capacity  = self.N / S

      district_votes = torch.zeros(B, S, K, device=alpha.device)

      for step in range(self.n_voters):
          party_logits = torch.log(alpha.clamp(min=eps))
          p_i          = _gumbel_softmax_st(party_logits, self.tau)

          district_totals = district_votes.sum(dim=2)
          vacancy_raw    = (capacity - district_totals).clamp(min=0.0)
          vac_sum        = vacancy_raw.sum(dim=1, keepdim=True) + eps
          vacancy        = vacancy_raw / vac_sum

          vv = (vacancy > 0).float().unsqueeze(1)
          vv = torch.transpose(vv, 1, 2)
          vv = vv.repeat(1, 1, K)
          #print(vv)
          district_sel = district_votes * vv
          #district_sel = district_votes

          party_totals = district_sel.sum(dim=1)
          denom        = party_totals.unsqueeze(1) + eps
          Ci           = district_sel / denom

          zero_mask  = (party_totals < eps).float().unsqueeze(1)
          Ci_uniform = torch.full((B, S, K), 1.0 / S, device=beta.device)
          Ci_uniform = district_sel * Ci_uniform
          Ci         = Ci * (1 - zero_mask) + Ci_uniform * zero_mask

          a              = beta.unsqueeze(1)
          district_probs = a * Ci + (1 - a) * vacancy.unsqueeze(2)

          q_i_logit   = (p_i.unsqueeze(1) * district_probs).sum(dim=2)
          dist_logits = torch.log(q_i_logit.clamp(min=eps))
          q_i_hard    = _gumbel_softmax_st(dist_logits, self.tau)
          increment      = p_i.unsqueeze(1) * q_i_hard.unsqueeze(2)
          district_votes = district_votes + increment

      vote_totals = district_votes.sum(dim=1)
    # Fix: Changed dim=2 to dim=1 for vote_totals.sum
      vote_share  = vote_totals / (vote_totals.sum(dim=1, keepdim=True) + eps)

      return vote_share, district_votes


# ============================================================
# FULL MODEL  (Encoder + GradABM)
# ============================================================

class GradABM(nn.Module):
    def __init__(self, N=5000, S=50, K=3,
                 hidden=128, n_voters=500, tau=1.0):
        super().__init__()
        self.encoder = EncoderNN2(num_districts=S, num_parties=K)
        self.sim     = GradABMSimulator(N=N, S=S, K=K,
                                        n_voters=n_voters, tau=tau)

    def forward(self, x):
        alpha, beta          = self.encoder(x)
        vote_share, phi = self.sim(alpha, beta)
        return vote_share, phi

In [41]:
def _has_nan_grad(model):
    for p in model.parameters():
        if p.grad is not None and torch.isnan(p.grad).any():
            return True
    return False


#def train_model(data, K=3, S=50, N=5000, n_voters=500,
#                epochs=30, batch_size=32, lr=2e-4,
#                tau=1.0,
#                lambda_kl=0.5, lambda_ce=0.25):

def train_encoder_decoder2(X, true_params, num_examples, num_districts, voters_per_district_in, voters_per_district_out, num_parties,
                          device='cpu', epochs=50, batch_size=8, lr=1e-3, tau=0.5,
                          gumbel_temp=0.5, verbose=True):

    N = num_districts * voters_per_district_out
    K = num_parties
    S = num_districts

    encoder_model = EncoderNN2(num_districts=S, num_parties=K).to(device)
    optimizer = optim.AdamW(encoder_model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    decoder_model = GradABMSimulator(N=N, S=S, K=K, n_voters=voters_per_district_out, tau=tau).to('cpu')

    mse_loss = nn.MSELoss()
    ce_loss = nn.CrossEntropyLoss()
    loss_history = {'total': [], 'reconstruction': [], 'variance': [], 'alpha': [], 'beta': [], 'theta': [], 'param_direct': []}
    best_loss = float('inf')

    alpha_true = torch.tensor(np.array([p[0] for p in true_params]), dtype=torch.float32, device=device)
    beta1_true = torch.tensor(np.array([p[1] for p in true_params]), dtype=torch.float32, device=device)
    beta2_true = torch.tensor(np.array([p[2] for p in true_params]), dtype=torch.float32, device=device)
    beta3_true = torch.tensor(np.array([p[3] for p in true_params]), dtype=torch.float32, device=device)
    theta_true = torch.tensor(np.array([p[4] for p in true_params]), dtype=torch.float32, device=device)

    start_time = time.time()
    indices = np.arange(num_examples)

    for ep in range(1, epochs + 1):
        encoder_model.train()
        np.random.shuffle(indices)
        epoch_loss_total = 0.0
        epoch_loss_recon = 0.0
        epoch_loss_alpha = 0.0
        epoch_loss_beta = 0.0
        epoch_loss_theta = 0.0
        epoch_loss_param = 0.0
        epoch_loss_variance = 0.0

        for i in range(0, num_examples, batch_size):
            batch_idx = indices[i:i+batch_size]
            X_range = []
            for j in batch_idx:
                x = X[j]
                distlist = np.random.choice(x.shape[0], num_districts, replace=False)
                x_sel = x[distlist]
                X_range.append(x_sel)

            phi_original = torch.tensor(np.stack(X_range), dtype=torch.float32, device=device)
            alpha_true_b = alpha_true[batch_idx]
            beta1_true_b = beta1_true[batch_idx]
            beta2_true_b = beta2_true[batch_idx]
            beta3_true_b = beta3_true[batch_idx]
            theta_true_b = theta_true[batch_idx]

            x_svar_true = district_share_variance(phi_original, num_districts)
            x_kvar_true = party_share_variance(phi_original, num_parties)
            #theta_true = party_seat_share(phi_original)

            # Encoder predicts alpha, beta
            alpha_pred, beta1_pred, beta2_pred, beta3_pred = encoder_model(phi_original)
            beta_pred = torch.stack((beta1_pred.squeeze(), beta2_pred.squeeze(), beta3_pred.squeeze()), dim=1)
            _, phi_reconstructed = decoder_model(alpha_true_b, beta_pred)

            x_svar_reconstructed = district_share_variance(phi_reconstructed, num_districts)
            x_kvar_reconstructed = party_share_variance(phi_reconstructed, num_parties)
            theta_reconstructed = party_seat_share(phi_reconstructed)

            loss_recon = mse_loss(phi_reconstructed / voters_per_district_out, phi_original / voters_per_district_in)
            loss_variance = mse_loss(x_svar_reconstructed, x_svar_true) + mse_loss(x_kvar_reconstructed, x_kvar_true)
            #loss_recon += loss_variance

            # 2. Direct parameter supervision (helps with gradient flow)
            target_cls = torch.argmax(alpha_true_b, dim=1)
            #loss_alpha1_direct = ce_loss(alpha_logits, target_cls)  #predict winning party
            #loss_alpha2_direct = ce_loss(alpha_true_b, alpha_pred)
            loss_alpha3_direct = mse_loss(alpha_true_b, alpha_pred)
            loss_beta1_direct = mse_loss(beta1_true_b, beta1_pred)    #predict ABM parameter
            loss_beta2_direct = mse_loss(beta2_true_b, beta2_pred)
            loss_beta3_direct = mse_loss(beta3_true_b, beta3_pred)
            loss_theta_direct = mse_loss(torch.tensor(theta_reconstructed, dtype=torch.float32, device=device), theta_true_b)

            #loss_param_direct = loss_alpha1_direct + loss_alpha2_direct + loss_alpha3_direct + loss_beta_direct + loss_theta_direct
            loss_param_direct = loss_theta_direct + 5*loss_beta1_direct + 5*loss_beta2_direct + 5*loss_beta3_direct + loss_alpha3_direct
            loss_beta_direct = loss_beta1_direct + loss_beta2_direct + loss_beta3_direct

            param_weight = 4
            recon_weight = 0
            variance_weight = 0

            loss = recon_weight * loss_recon + param_weight * loss_param_direct + variance_weight * loss_variance

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(encoder_model.parameters(), max_norm=1.0)
            optimizer.step()

            # Track losses
            epoch_loss_total += loss.item() * phi_original.shape[0]
            epoch_loss_recon += loss_recon.item() * phi_original.shape[0]
            epoch_loss_variance += loss_variance.item() * phi_original.shape[0]
            epoch_loss_param += loss_param_direct.item() * phi_original.shape[0]
          # epoch_loss_alpha += loss_alpha1_direct.item() * phi_original.shape[0]
          # epoch_loss_alpha += loss_alpha2_direct.item() * phi_original.shape[0]
            epoch_loss_alpha += loss_alpha3_direct.item() * phi_original.shape[0]
            epoch_loss_beta += loss_beta_direct.item() * phi_original.shape[0]
            epoch_loss_theta += loss_theta_direct.item() * phi_original.shape[0]

          # Average losses
        epoch_loss_total /= num_examples
        epoch_loss_recon /= num_examples
        epoch_loss_variance /= num_examples
        epoch_loss_param /= num_examples
        epoch_loss_alpha /= num_examples
        epoch_loss_beta /= num_examples
        epoch_loss_theta /= num_examples

        loss_history['total'].append(epoch_loss_total)
        loss_history['reconstruction'].append(epoch_loss_recon)
        loss_history['variance'].append(epoch_loss_variance)
        loss_history['param_direct'].append(epoch_loss_param)
        loss_history['alpha'].append(epoch_loss_alpha)
        loss_history['beta'].append(epoch_loss_beta)
        loss_history['theta'].append(epoch_loss_theta)

        # Update learning rate
        scheduler.step(epoch_loss_total)

        #if verbose and (ep % 10 == 0 or ep == 1):
        if verbose :
                print(f"Epoch {ep:04d} | Total: {epoch_loss_total:.4f} | "
                  f"Recon: {epoch_loss_recon:.4f} | Param: {epoch_loss_param:.4f} | "
                  f"Variance: {epoch_loss_variance:.4f} | "
                  f"Alpha: {epoch_loss_alpha:.4f} | Beta: {epoch_loss_beta:.4f} | "
                  f"Theta: {epoch_loss_theta:.4f} | "
                  f"LR: {optimizer.param_groups[0]['lr']:.6f} | "
                  f"elapsed {time.time()-start_time:.1f}s")

        # Save best model
        if epoch_loss_total < best_loss:
            best_loss = epoch_loss_total

        # Create X_tensor from the training data X (which has num_examples) for correlation calculation
        X_tensor_for_corr = torch.tensor(np.stack(X), dtype=torch.float32, device=device)
        alpha_pred_all, beta1_pred_all, beta2_pred_all, beta3_pred_all = encoder_model(X_tensor_for_corr)
        #c1 = np.correlate(alpha_pred_all.cpu().numpy(), alpha_true.cpu().numpy())
        c1 = scipy.stats.pearsonr(beta1_pred_all.detach().cpu().numpy().squeeze(), beta1_true.cpu().numpy().squeeze())[0]
        c2 = scipy.stats.pearsonr(beta2_pred_all.detach().cpu().numpy().squeeze(), beta2_true.cpu().numpy().squeeze())[0]
        c3 = scipy.stats.pearsonr(beta3_pred_all.detach().cpu().numpy().squeeze(), beta3_true.cpu().numpy().squeeze())[0]
        print("c1=", c1, "c2=", c2, "c3=", c3)

    return encoder_model, decoder_model, loss_history

In [42]:
import scipy
import scipy.io as sio
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)
mat_file_path = '/content/drive/MyDrive/indiasurveys_aug_PCM.mat'
data=sio.loadmat(mat_file_path)

X = data['CC']
alpha = data['alpha']
beta1 = data['beta1']
beta2 = data['beta2']
beta3 = data['beta3']
theta = data['theta']
print(X.shape)
print(alpha.shape)
print(beta1.shape)
x= X[0][10]
print(x.shape)
print(alpha[500])
print(beta1[0])

NUM_DISTRICTS = X[0][0].shape[0]
NUM_PARTIES = X[0][0].shape[1]
VOTERS_PER_DISTRICT_IN = 100 #int(np.sum(X[0][0][0]))
VOTERS_PER_DISTRICT_OUT = 10000 #int(np.sum(X[0][0][0]))

#voters_scaling = 100
#X = X/voters_scaling
#VOTERS_PER_DISTRICT = int(VOTERS_PER_DISTRICT/voters_scaling)
print(VOTERS_PER_DISTRICT)

train_range = range(0,2500)
NUM_EXAMPLES = len(train_range)

X_np, Y, true_params = [], [], []
for i in train_range:
      X_np.append(X[0][i].astype(np.float32))
      # Fix: Removed float() conversion as beta[i] is an array, not a scalar
      Y.append((alpha[i].astype(np.float32), beta1[i].astype(np.float32), beta2[i].astype(np.float32), beta3[i].astype(np.float32)))
      true_params.append((alpha[i].astype(np.float32), beta1[i].astype(np.float32), beta2[i].astype(np.float32), beta3[i].astype(np.float32), theta[i].astype(np.float32)))

Mounted at /content/drive
(1, 8000)
(8000, 3)
(5000, 1)
(10, 3)
[0.23825763 0.47528606 0.28645632]
[0.57225825]
100


In [43]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import RelaxedOneHotCategorical
import scipy.stats
import matplotlib.pyplot
import gc # Import garbage collector

from scipy.io import savemat
import numpy as np

EPOCHS = 35
BATCH_SIZE = 500  # Reduced batch size to mitigate OutOfMemoryError

#if __name__ == "__main__":
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

# Clear GPU memory before starting training, if available
if device == 'cuda':
    print("Clearing GPU memory...")
    torch.cuda.empty_cache()
    gc.collect() # Force Python garbage collection
    print("GPU memory cleared.")
    print("GPU memory summary AFTER clearing cache:")
    print(torch.cuda.memory_summary())

print(f"VOTERS_PER_DISTRICT: {VOTERS_PER_DISTRICT}")
print("Training on GPU..." if device == 'cuda' else "Training on CPU...")
encoder_PCM, decoder_PCM, loss_history = train_encoder_decoder2(
      X_np, true_params,
      num_examples=NUM_EXAMPLES,
      num_districts=NUM_DISTRICTS,
      num_parties=NUM_PARTIES,
      voters_per_district_in=VOTERS_PER_DISTRICT_IN,
      voters_per_district_out=VOTERS_PER_DISTRICT_OUT,
      device=device,
      epochs=EPOCHS,
      batch_size=BATCH_SIZE,
      lr=1e-3,
      gumbel_temp=0.75
    )

print("\nEncoder-decoder training complete.")

Device: cuda
Clearing GPU memory...
GPU memory cleared.
GPU memory summary AFTER clearing cache:
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |  29519 KiB |   3837 MiB |   3138 GiB |   3138 GiB |
|       from large pool |  16640 KiB |     60 MiB |      4 GiB |      4 GiB |
|       from small pool |  12879 KiB |   3810 MiB |   3134 GiB |   3134 GiB |
|---------------------------------------------------------------------------|
| Active memory         |  29519 KiB |   3837

In [46]:
Xtest = data['CC']

X_test_processed = []
for i in range(Xtest[0].shape[0]): # Iterate through the actual number of samples in Xtest_raw[0]
    x = Xtest[0][i].astype(np.float32)
    X_test_processed.append(x)

X_tensor = torch.tensor(np.stack(X_test_processed), dtype=torch.float32, device=device)
grad_alpha_pred, grad_beta1_pred, grad_beta2_pred, grad_beta3_pred = encoder_PCM(X_tensor)
grad_alpha_pred = grad_alpha_pred.detach().cpu().numpy()
grad_beta1_pred = grad_beta1_pred.detach().cpu().numpy()
grad_beta2_pred = grad_beta2_pred.detach().cpu().numpy()
grad_beta3_pred = grad_beta3_pred.detach().cpu().numpy()

print(alpha[4000])
print(beta1[4000])
print(beta2[4000])
print(beta3[4000])
print(grad_alpha_pred[4000])
print(grad_beta1_pred[4000])
print(grad_beta2_pred[4000])
print(grad_beta3_pred[4000])

print(grad_alpha_pred[5499])
print(grad_alpha_pred[5999])
print(grad_alpha_pred[6499])
print(grad_alpha_pred[6999])
print(grad_alpha_pred[7499])
print(grad_alpha_pred[7999])

[0.35591875 0.38981175 0.2542695 ]
[0.60252798]
[0.65683023]
[0.65477118]
[0.33919474 0.38553536 0.27526987]
[0.6550404]
[0.6472711]
[0.6211023]
[0.38698024 0.4103782  0.20264158]
[0.5569984  0.23298647 0.21001518]
[0.30343577 0.5661908  0.13037345]
[0.37989682 0.31864288 0.30146036]
[0.4119316  0.42327192 0.16479641]
[0.31633848 0.4893508  0.19431068]


In [50]:
from scipy.io import savemat
import numpy as np
PCM_pred = {"CC": X, "alpha": alpha, "beta1": beta1, "beta2": beta2, "beta3": beta3, "theta": theta, "alpha1": alpha_pred, "beta1_pred2": beta1_pred, "beta2_pred2": beta2_pred, "beta3_pred2": beta3_pred,  "alpha_pred1": grad_alpha_pred, "beta1_pred1": grad_beta1_pred, "beta2_pred1": grad_beta2_pred, "beta3_pred1": grad_beta3_pred}
savemat("PCM_pred.mat", PCM_pred)